In [22]:
import os
import re
import time
import mimetypes
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from urllib.parse import urlparse
from datetime import datetime, timezone
import csv
import pandas as pd
import requests

In [14]:

# =========================
# CONFIG
# =========================
INPUT_POSTS_CSV = "Instagram/ig_post_info.csv"
OUT_DIR = Path("Instagram/posts_info/Downloads")

# persistent job log (append over time)
JOB_LOG_CSV = "Instagram/posts_downloads_log.csv"

SLEEP_SECONDS = 0.2
TIMEOUT = 60
MAX_RETRIES = 2

In [5]:
# =========================
# Helpers
# =========================
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def make_run_id() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

def safe_filename(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", "_", s)
    return re.sub(r"[^a-zA-Z0-9._-]", "", s)

def split_urls(s: str) -> List[str]:
    if s is None:
        return []
    s = str(s).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return []
    parts = [p.strip() for p in s.split(",")]
    return [p for p in parts if p]

def is_valid_http_url(u: str) -> bool:
    if u is None:
        return False
    u = str(u).strip()
    if not u or u.lower() in {"nan", "none", "null"}:
        return False
    return u.startswith("http://") or u.startswith("https://")

def guess_ext_from_url(url: str) -> str:
    path = urlparse(url).path
    ext = Path(path).suffix.lower()
    if ext and len(ext) <= 5:
        return ext
    return ".bin"

def guess_ext_from_content_type(content_type: Optional[str], fallback: str = ".bin") -> str:
    if not content_type:
        return fallback
    content_type = content_type.split(";")[0].strip().lower()
    ext = mimetypes.guess_extension(content_type) or fallback
    if ext == ".jpe":
        ext = ".jpg"
    return ext

In [6]:
# =========================
# Build tasks (now includes post_type)
# =========================
def build_tasks(df: pd.DataFrame) -> List[Dict]:
    tasks: List[Dict] = []

    for col in ["post_id", "post_type", "carousel_media_urls", "video_url", "cover_url"]:
        if col not in df.columns:
            df[col] = ""

    for _, r in df.iterrows():
        post_id = str(r.get("post_id", "")).strip()
        if not post_id:
            continue

        post_type = str(r.get("post_type", "")).strip()
        carousel_urls = split_urls(r.get("carousel_media_urls", ""))
        video_url = str(r.get("video_url", "") or "").strip()
        cover_url = str(r.get("cover_url", "") or "").strip()

        if post_type == "GraphSidecar" and carousel_urls:
            for i, u in enumerate(carousel_urls, start=1):
                if not is_valid_http_url(u):
                    continue
                tasks.append({
                    "post_id": post_id,
                    "post_type": post_type,
                    "slide_number": i,
                    "url": u,
                    "kind": "carousel_slide",
                    "file_title_base": f"{safe_filename(post_id)}_{i}",
                })

        elif post_type == "GraphVideo" and is_valid_http_url(video_url):
            tasks.append({
                "post_id": post_id,
                "post_type": post_type,
                "slide_number": 1,
                "url": video_url,
                "kind": "video",
                "file_title_base": f"{safe_filename(post_id)}",
            })

        else:
            # GraphImage or fallback: cover_url
            if not is_valid_http_url(cover_url):
                continue
            tasks.append({
                "post_id": post_id,
                "post_type": post_type,
                "slide_number": 1,
                "url": cover_url,
                "kind": "cover_image",
                "file_title_base": f"{safe_filename(post_id)}",
            })

    return tasks

In [7]:
# =========================
# Persistent job log load + skip logic
# =========================
def load_job_log(job_log_csv: str) -> pd.DataFrame:
    if not Path(job_log_csv).exists():
        return pd.DataFrame(columns=[
            "post_id","post_type","slide_number","kind","url",
            "downloaded","saved_filename","local_path","error_type","error_message"
        ])
    return pd.read_csv(job_log_csv)

def task_key(task: Dict) -> Tuple[str, int, str, str]:
    # identity of a task
    return (str(task["post_id"]), int(task["slide_number"]), str(task["kind"]), str(task["url"]))

def filter_tasks_skip_success(tasks: List[Dict], job_log_df: pd.DataFrame) -> List[Dict]:
    """
    Skip tasks that have previously succeeded for same (post_id, slide_number, kind, url).
    """
    if job_log_df.empty:
        return tasks

    # treat any downloaded==True as success
    # (make robust to "True"/"False" strings)
    jl = job_log_df.copy()
    if "downloaded" in jl.columns:
        jl["downloaded"] = jl["downloaded"].astype(str).str.lower().isin(["true", "1", "yes"])

    success = jl[jl.get("downloaded", False) == True]
    if success.empty:
        return tasks

    success_keys = set(
        zip(
            success["post_id"].astype(str),
            success["slide_number"].astype(int),
            success["kind"].astype(str),
            success["url"].astype(str),
        )
    )

    remaining = [t for t in tasks if task_key(t) not in success_keys]
    return remaining

In [8]:
# =========================
# Download (returns richer info)
# =========================
def download_one(session: requests.Session, url: str, out_path_no_ext: Path) -> Dict:
    """
    Returns dict with:
      ok(bool), status_code(int|None), saved_path(str|""), saved_filename(str|""),
      error_type(str|""), error_message(str|""), attempts(int), duration_sec(float)
    """
    t0 = time.time()

    url_ext = guess_ext_from_url(url)
    final_path = out_path_no_ext.with_suffix(url_ext)

    # If already exists locally, treat as success
    if final_path.exists() and final_path.stat().st_size > 0:
        return {
            "ok": True,
            "status_code": None,
            "saved_path": str(final_path),
            "saved_filename": final_path.name,
            "error_type": "",
            "error_message": "",
            "attempts": 0,
            "duration_sec": round(time.time() - t0, 4),
        }

    last_exc = None
    last_status = None

    for attempt in range(1, MAX_RETRIES + 2):  # total attempts = MAX_RETRIES+1
        try:
            with session.get(url, stream=True, timeout=TIMEOUT, allow_redirects=True) as resp:
                last_status = resp.status_code

                # Common failure buckets
                if resp.status_code == 429:
                    raise requests.HTTPError("HTTP 429 (rate limited)")
                if resp.status_code in (403, 404):
                    raise requests.HTTPError(f"HTTP {resp.status_code} (expired/private?)")

                resp.raise_for_status()

                # Infer ext from content-type if URL ext unknown
                if url_ext == ".bin":
                    ext2 = guess_ext_from_content_type(resp.headers.get("Content-Type"), fallback=".bin")
                    final_path = out_path_no_ext.with_suffix(ext2)

                if final_path.exists() and final_path.stat().st_size > 0:
                    return {
                        "ok": True,
                        "status_code": last_status,
                        "saved_path": str(final_path),
                        "saved_filename": final_path.name,
                        "error_type": "",
                        "error_message": "",
                        "attempts": attempt,
                        "duration_sec": round(time.time() - t0, 4),
                    }

                OUT_DIR.mkdir(parents=True, exist_ok=True)
                tmp_path = final_path.with_suffix(final_path.suffix + ".part")

                with tmp_path.open("wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 256):
                        if chunk:
                            f.write(chunk)

                tmp_path.replace(final_path)

                return {
                    "ok": True,
                    "status_code": last_status,
                    "saved_path": str(final_path),
                    "saved_filename": final_path.name,
                    "error_type": "",
                    "error_message": "",
                    "attempts": attempt,
                    "duration_sec": round(time.time() - t0, 4),
                }

        except Exception as e:
            last_exc = e
            # backoff (especially useful for 429)
            time.sleep(0.5 * attempt)

    err_type = type(last_exc).__name__ if last_exc else "UnknownError"
    err_msg = str(last_exc) if last_exc else "Unknown error"

    return {
        "ok": False,
        "status_code": last_status,
        "saved_path": "",
        "saved_filename": "",
        "error_type": err_type,
        "error_message": err_msg,
        "attempts": MAX_RETRIES + 1,
        "duration_sec": round(time.time() - t0, 4),
    }


In [27]:
# =========================
# Append log rows safely
# =========================
JOB_LOG_COLUMNS = [
    "run_id",
    "run_started_at",
    "run_finished_at",
    "post_id",
    "post_type",
    "slide_number",
    "kind",
    "url",
    "downloaded",
    "status_code",
    "error_type",
    "error_message",
    "saved_filename",
    "local_path",
    "attempts",
    "duration_sec",
    "error_bucket",
]

def append_job_log(job_log_csv: str, rows: List[Dict]) -> None:
    """
    Append rows to a CSV with a FIXED schema so columns never shift.
    """
    if not rows:
        return

    out_path = Path(job_log_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    file_exists = out_path.exists() and out_path.stat().st_size > 0

    with out_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=JOB_LOG_COLUMNS, extrasaction="ignore")
        if not file_exists:
            writer.writeheader()

        for r in rows:
            # enforce fixed schema + fill missing keys with ""
            writer.writerow({k: r.get(k, "") for k in JOB_LOG_COLUMNS})

        f.flush()
        try:
            import os
            os.fsync(f.fileno())
        except Exception:
            pass



# =========================
# Runner (skip success, retry failures)
# =========================
def run_downloads_with_joblog(
    input_posts_csv: str = INPUT_POSTS_CSV,
    out_dir: Path = OUT_DIR,
    job_log_csv: str = JOB_LOG_CSV,
    sleep_seconds: float = SLEEP_SECONDS,
) -> pd.DataFrame:
    """
    Runs downloads and APPENDS one log row to job_log_csv after EACH task completes.
    Skips tasks that were previously downloaded successfully.
    Returns a DataFrame containing ONLY the rows from this run.
    """
    global OUT_DIR
    OUT_DIR = out_dir

    run_id = make_run_id()
    run_started_at = utc_now_iso()

    df_posts = pd.read_csv(input_posts_csv)
    tasks = build_tasks(df_posts[100:105])

    # load log and skip prior successes
    job_log_df = load_job_log(job_log_csv)
    tasks_to_run = filter_tasks_skip_success(tasks, job_log_df)

    print(f"Total tasks in input: {len(tasks)}")
    print(f"Skipping previously successful: {len(tasks) - len(tasks_to_run)}")
    print(f"Tasks to run now: {len(tasks_to_run)}")

    run_rows: List[Dict] = []

    with requests.Session() as session:
        session.headers.update({
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                          "(KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
            "Referer": "https://www.instagram.com/",
            "Origin": "https://www.instagram.com",
            "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
            "Connection": "keep-alive",
        })

        for idx, t in enumerate(tasks_to_run, start=1):
            post_id = t["post_id"]
            post_type = t.get("post_type", "")
            slide_number = int(t["slide_number"])
            url = t["url"]
            kind = t["kind"]
            file_base = t["file_title_base"]

            out_path_no_ext = OUT_DIR / file_base

            result = download_one(session, url, out_path_no_ext)

            # optional: categorize errors (handy for analysis)
            error_bucket = ""
            if not result["ok"]:
                sc = result["status_code"]
                if sc in (403, 404):
                    error_bucket = "expired_or_private"
                elif sc == 429:
                    error_bucket = "rate_limited"
                elif result["error_type"] in ("MissingSchema", "InvalidURL"):
                    error_bucket = "bad_url"
                else:
                    error_bucket = "other"

            row = {
                "run_id": run_id,
                "run_started_at": run_started_at,
                "run_finished_at": "",  # we'll fill at the end, but log the row now anyway
                "post_id": post_id,
                "post_type": post_type,
                "slide_number": slide_number,
                "kind": kind,
                "url": url,
                "downloaded": result["ok"],
                "status_code": result["status_code"],
                "error_bucket": error_bucket,
                "error_type": result["error_type"],
                "error_message": result["error_message"],
                "saved_filename": result["saved_filename"],
                "local_path": result["saved_path"],
                "attempts": result["attempts"],
                "duration_sec": result["duration_sec"],
            }

            # ✅ SAVE LOG IMMEDIATELY (crash-safe)
            append_job_log(job_log_csv, [row])

            run_rows.append(row)

            print(
                f"[{idx}/{len(tasks_to_run)}] post_id={post_id} slide={slide_number} "
                f"kind={kind} ok={result['ok']} status={result['status_code']} "
                f"{('err=' + row['error_type']) if not result['ok'] else ''}"
            )

            time.sleep(sleep_seconds)

    # Finish: mark run_finished_at for the in-memory df we return
    run_finished_at = utc_now_iso()
    for r in run_rows:
        r["run_finished_at"] = run_finished_at

    # Note: we did NOT update past rows in the CSV to add run_finished_at,
    # because we append-per-task. If you really want it, we can add a separate
    # "run summary" row or a separate run_summary.csv.

    return pd.DataFrame(run_rows)


In [28]:

if __name__ == "__main__":
    run_downloads_with_joblog(
        input_posts_csv=INPUT_POSTS_CSV,
        out_dir=OUT_DIR,
        job_log_csv=JOB_LOG_CSV,
        sleep_seconds=SLEEP_SECONDS,
    )


Total tasks in input: 7
Skipping previously successful: 0
Tasks to run now: 7
[1/7] post_id=3823732431648645952 slide=1 kind=video ok=False status=403 err=HTTPError
[2/7] post_id=3822906186988082273 slide=1 kind=video ok=False status=403 err=HTTPError
[3/7] post_id=3822083696909151039 slide=1 kind=carousel_slide ok=False status=403 err=HTTPError
[4/7] post_id=3822083696909151039 slide=2 kind=carousel_slide ok=False status=403 err=HTTPError
[5/7] post_id=3822083696909151039 slide=3 kind=carousel_slide ok=False status=403 err=HTTPError
[6/7] post_id=3819170466335151166 slide=1 kind=video ok=False status=403 err=HTTPError
[7/7] post_id=3818718633284027186 slide=1 kind=video ok=False status=403 err=HTTPError
